# Fix Neural Data from MATLAB Files

This notebook loads sorted spikes data from MATLAB files and creates a DataFrame with trial names and neural data.

In [1]:
import numpy as np
import pandas as pd
import re
from scipy.io import loadmat
from pathlib import Path
from tqdm import tqdm

monkey_name = 'fiona' # 'fiona' or 'yasmin'

In [2]:
def parse_neural_data_from_mat(data):
    """
    Parse MATLAB dataStruct into a DataFrame with trial names and neural data.
    
    Parameters:
    -----------
    data : dict
        Dictionary loaded from MATLAB file using scipy.io.loadmat
    
    Returns:
    --------
    pd.DataFrame : DataFrame with columns ['trial_name', 'neural_data']
        neural_data is a dict with keys=neuron_id (int), values=spike_times (numpy array)
    """
    data_struct = data['dataStruct']
    num_trials = data_struct.shape[1]
    
    trial_names = []
    neural_data_list = []
    
    for i in range(num_trials):
        trial_entry = data_struct[0, i]
        
        # Extract trial name
        trial_name_obj = trial_entry[0]
        if hasattr(trial_name_obj, '__len__') and len(trial_name_obj) > 0:
            trial_name = str(trial_name_obj[0])
        else:
            raise ValueError(f"Unexpected trial name structure in trial index {i}")
            # trial_name = f"trial_{i}"
        
        # Extract spikes for all neurons
        spikes_obj = trial_entry[1]
        neural_data_dict = {}
        
        if hasattr(spikes_obj, 'shape') and len(spikes_obj.shape) > 1:
            for idx, neuron_spikes in enumerate(spikes_obj[0]):
                # Flatten the spike times array and add them to the dict
                neural_data_dict[idx] = neuron_spikes.flatten()
            # num_neurons = spikes_obj.shape[1]
            
            # for j in range(num_neurons):
            #     neuron_spikes = spikes_obj[0, j]
                
            #     # Flatten the spike times array
            #     if hasattr(neuron_spikes, 'flatten'):
            #         spike_times = neuron_spikes.flatten()
            #         # Only add non-empty spike trains
            #         if len(spike_times) > 0:
            #             neural_data_dict[j] = spike_times
            #     elif hasattr(neuron_spikes, '__len__') and len(neuron_spikes) > 0:
            #         spike_times = np.array(neuron_spikes).flatten()
            #         if len(spike_times) > 0:
            #             neural_data_dict[j] = spike_times
            #     else:
            #         raise ValueError(f"Unexpected spike times structure for neuron {j} in trial {trial_name}")
        else:
            raise ValueError(f"Unexpected spikes object structure in trial {trial_name}")
        trial_names.append(trial_name)
        neural_data_list.append(neural_data_dict)

    
    # Create DataFrame
    df = pd.DataFrame({
        'trial_name': trial_names,
        'neural_data': neural_data_list
    })
    
    return df


In [3]:
# Load the exemplar file
data_dir = Path.cwd().parent / 'data' / 'fiona_sst' / 'sorted_spikes_in_session'
mat_file = data_dir / 'fi211109_sorted_spikes.mat'

print(f"Loading: {mat_file}")
print(f"File exists: {mat_file.exists()}")

if mat_file.exists():
    data = loadmat(mat_file)
    print(f"\nTop-level keys: {list(data.keys())}")
    print(f"\ndataStruct shape: {data['dataStruct'].shape}")
    print(f"dataStruct dtype: {data['dataStruct'].dtype}")
    print(f"\nNumber of trials: {data['dataStruct'].shape[1]}")


# Parse examplar data
neural_df = parse_neural_data_from_mat(data)

print(f"Created DataFrame with shape: {neural_df.shape}")
print(f"\nFirst few rows:")
print(neural_df.head())
print(f"\nSample neural_data for first trial:")
first_trial_data = neural_df.iloc[0]['neural_data']
print(f"  Number of neurons with spikes: {len(first_trial_data)}")
print(f"  Neuron IDs: {list(first_trial_data.keys())[:10]}...")  # Show first 10
if len(first_trial_data) > 0:
    first_neuron_id = list(first_trial_data.keys())[0]
    print(f"  Example - Neuron {first_neuron_id} spike times: {first_trial_data[first_neuron_id]}")

Loading: /home/barak/Projects/population-analysis/data/fiona_sst/sorted_spikes_in_session/fi211109_sorted_spikes.mat
File exists: True

Top-level keys: ['__header__', '__version__', '__globals__', 'dataStruct']

dataStruct shape: (1, 2040)
dataStruct dtype: [('trial', 'O'), ('spikes', 'O')]

Number of trials: 2040
Created DataFrame with shape: (2040, 2)

First few rows:
       trial_name                                        neural_data
0  fi211109a.0001  {0: [], 1: [1510.14, 1632.64], 2: [], 3: [], 4...
1  fi211109a.0002  {0: [1196.53], 1: [455.41, 1398.74, 1966.24, 2...
2  fi211109a.0003  {0: [1229.15, 1318.25], 1: [961.06], 2: [], 3:...
3  fi211109a.0004  {0: [2339.12], 1: [326.66, 972.53], 2: [], 3: ...
4  fi211109a.0005  {0: [1033.38, 1055.85], 1: [875.09, 2149.34], ...

Sample neural_data for first trial:
  Number of neurons with spikes: 200
  Neuron IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...
  Example - Neuron 0 spike times: []


In [4]:
data_struct = data['dataStruct']
num_trials = data_struct.shape[1]

trial_entry = data_struct[0, 0]
trial_name_obj = trial_entry[0]
spikes_obj = trial_entry[1]
num_neurons = spikes_obj.shape[1]

neuron_spikes = spikes_obj[0, 0]

spike_times = neuron_spikes.flatten()

spikes_obj[0][0]

array([], shape=(1, 0), dtype=float64)

In [5]:
# Aggregate neural data from all '*_sorted_spikes.mat' files under data/

data_root = Path.cwd().parent / 'data' / f'{monkey_name}_sst' / 'sorted_spikes_in_session'
pattern = f'{monkey_name[:2]}2*_sorted_spikes.mat'
all_files = sorted(list(data_root.rglob(pattern)))
print(f'Found {len(all_files)} files matching pattern "{pattern}" under {data_root}')

# Filter out hidden/system files (starting with ._) and validate session name format
session_pattern = re.compile(r'^(?:fi|ya)2\d{5}$')
valid_files = []
for f in all_files:
    session_name = f.stem.split('_sorted_spikes')[0]
    if ((f.name.startswith('._')) or (not session_pattern.match(session_name))):
        print(f'Skipping hidden/system file: {f.name}')
        continue
    valid_files.append(f)

print(f'\nProcessing {len(valid_files)} valid files of {len(all_files)} files found:\n')

dfs = []
for mat_file in tqdm(valid_files, desc='Processing files'):
    # print(f'Processing: {mat_file.relative_to(data_root)}')
    try:
        data_local = loadmat(mat_file)
    except Exception as e:
        print(f'  Failed to load {mat_file.name}: {e}')
        continue

    # Parse using the function defined earlier in this notebook
    df = parse_neural_data_from_mat(data_local)

    # Derive session name from filename (strip suffix '_sorted_spikes')
    session_name = mat_file.stem.split('_sorted_spikes')[0]
    df['session'] = session_name

    # Ensure trial_number exists as integer on each parsed df
    if 'trial_number' not in df.columns:
        df['trial_number'] = df['trial_name'].apply(lambda x: int(x.split('.')[-1]))
    else:
        # coerce to int if strings with leading zeros are present
        df['trial_number'] = df['trial_number'].astype(int)

    dfs.append(df)

# Concatenate all session DataFrames
if len(dfs) > 0:
    neural_df_all = pd.concat(dfs, ignore_index=True)
    print(f'Combined DataFrame shape: {neural_df_all.shape}')
    print(f'Total sessions: {neural_df_all["session"].unique().shape[0]}')
    print(f'Total trials: {len(neural_df_all)}')
else:
    print('No parsed files found; neural_df_all not created')

Found 88 files matching pattern "fi2*_sorted_spikes.mat" under /home/barak/Projects/population-analysis/data/fiona_sst/sorted_spikes_in_session

Processing 88 valid files of 88 files found:



Processing files: 100%|██████████| 88/88 [00:13<00:00,  6.57it/s]

Combined DataFrame shape: (126165, 4)
Total sessions: 88
Total trials: 126165


In [7]:
len(valid_files)

88

In [18]:
print("Neurons per trial: ",
    neural_df_all.apply(
        lambda row: len(row['neural_data'].keys()), 
        axis=1
    ).value_counts().index[0]
)
neural_df_all.head()

Neurons per trial:  200


,trial_name,neural_data,session,trial_number
0,fi210628a.0001,"{0: [28.44, 60.23, 94.46, 164.23, 193.88, 223....",fi210628,1
1,fi210628a.0002,"{0: [18.74, 59.81, 90.46, 131.34, 170.34, 202....",fi210628,2
2,fi210628a.0003,"{0: [5.49, 88.89, 178.81, 380.84, 480.36, 608....",fi210628,3
3,fi210628a.0004,"{0: [70.66, 437.66, 656.51, 696.26, 754.99, 81...",fi210628,4
4,fi210628a.0005,"{0: [66.44, 127.21, 154.31, 210.86, 235.24, 26...",fi210628,5


In [19]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

neural_df_all.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}')

Total unique neurons across all sessions: 17600


In [24]:
1307+1794+579+113

3793

In [25]:
base_path = Path.cwd().parent / 'data' 
file_path = base_path / 'csst_trials_pkls' / f'all_{monkey_name}_CSST_trials_df.pkl'

orig_df = pd.read_pickle(file_path)
orig_df

,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...","{0: [884.4], 1: [154.42, 329.18, 1478.9], 3: [...",...,2.0,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4...."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....","{1: [48.52, 264.55, 585.97, 1032.3], 2: [385.2...",...,NaN,NaN,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...","{1: [574.7, 853.02, 1403.57], 3: [8.35, 301.92...",...,3.0,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...","{2: [1954.73], 5: [434.05, 484.13, 547.7, 851....",...,3.0,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...","{0: [923.35], 1: [1125], 5: [1026.4, 1359.92, ...",...,2.0,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110353,None,0,R,fi211020a.0230,"[1213, 1285]",8206,943,"[-9.925, -9.925, -9.925, -9.925, -9.95, -9.95,...","[-0.09188980574495066, -0.09188980574495066, -...","{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7...",...,NaN,NaN,False,2094,GO_R,0230,fi211020a,GO,"[-1.35, -1.35, -1.45, -1.45, -1.35, -1.35, -1....","[-1.8377961148990132, -1.8377961148990132, -3...."
110354,None,0,R,fi211020a.0832,"[1397, 1470]",8206,1070,"[-11.925, -11.925, -11.85, -11.85, -11.875, -1...","[-4.686380092992484, -4.686380092992484, 0.0, ...","{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5...",...,NaN,NaN,False,2221,GO_R,0832,fi211020a,GO,"[-0.7, -0.7, -0.625, -0.625, -0.675, -0.675, -...","[-3.3080330068182238, -3.3080330068182238, 0.2..."
110355,None,180,L,fi211020a.1304,"[1210, 1287]",8206,1051,"[-11.95, -11.95, -11.95, -11.95, -11.95, -11.9...","[2.389134949368717, 0.5513388344697039, 0.5513...","{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7...",...,NaN,NaN,False,2202,GO_L,1304,fi211020a,GO,"[0.725, 0.75, 0.75, 0.775, 0.75, 0.75, 0.725, ...","[-1.6540165034091119, -1.1026776689394078, -1...."
110356,None,180,L,fi211020a.0719,"[1123, 1197]",8194,1001,"[0.0, 0.0, -0.1, -0.175, -0.175, -0.175, -0.37...","[-51.090731994192566, -51.090731994192566, -50...","{0: [521.64, 535.58, 677.96, 741.66, 963.88, 9...",...,NaN,NaN,True,2057,GO_L,0719,fi211020a,GO,"[-4.425, -4.425, -3.35, -3.0, -2.65, -2.65, -1...","[188.098432359914, 188.098432359914, 188.09843..."


In [26]:
cell_ids = set([])
cell_ids_list = orig_df['neural_data'].apply(
    lambda x: list(x.keys()) if isinstance(x, dict) else x
)

for row in cell_ids_list:
    if isinstance(row, list):
        cell_ids.update(row)

lst = [i for i in cell_ids]
lst.sort()
lst == list(range(0, 50))

False

In [28]:
# Join neural_df_all with orig_df
# Select only trial_name and neural_data from neural_df_all
neural_subset = neural_df_all[['trial_name', 'neural_data']].copy()

# Rename neural_data to new_neural_data before merging
neural_subset.rename(columns={'neural_data': 'new_neural_data'}, inplace=True)

# Merge on trial_name (neural_df_all) = filename (orig_df)
merged_df = orig_df.merge(
    neural_subset,
    left_on='filename',
    right_on='trial_name',
    how='left'
)

print(f"Original df shape: {orig_df.shape}")
print(f"Neural df shape: {neural_df_all.shape}")
print(f"Merged df shape: {merged_df.shape}")
print(f"\nMerged df columns: {list(merged_df.columns)}")
print(f"\nRows with new_neural_data: {merged_df['new_neural_data'].notna().sum()}")
print(f"Rows without new_neural_data: {merged_df['new_neural_data'].isna().sum()}")

merged_df

Original df shape: (110358, 28)
Neural df shape: (126165, 4)
Merged df shape: (110358, 30)

Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'trial_name_y', 'new_neural_data']

Rows with new_neural_data: 110358
Rows without new_neural_data: 0


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,trial_failed,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,trial_name_y,new_neural_data
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...","{0: [884.4], 1: [154.42, 329.18, 1478.9], 3: [...",...,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4....",fi210824a.0614,"{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [..."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....","{1: [48.52, 264.55, 585.97, 1032.3], 2: [385.2...",...,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1...",fi210824a.0520,"{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...","{1: [574.7, 853.02, 1403.57], 3: [8.35, 301.92...",...,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4...",fi210824a.1193,"{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...","{2: [1954.73], 5: [434.05, 484.13, 547.7, 851....",...,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ...",fi210824a.1013,"{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...","{0: [923.35], 1: [1125], 5: [1026.4, 1359.92, ...",...,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0...",fi210824a.1257,"{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110353,None,0,R,fi211020a.0230,"[1213, 1285]",8206,943,"[-9.925, -9.925, -9.925, -9.925, -9.95, -9.95,...","[-0.09188980574495066, -0.09188980574495066, -...","{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7...",...,False,2094,GO_R,0230,fi211020a,GO,"[-1.35, -1.35, -1.45, -1.45, -1.35, -1.35, -1....","[-1.8377961148990132, -1.8377961148990132, -3....",fi211020a.0230,"{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7..."
110354,None,0,R,fi211020a.0832,"[1397, 1470]",8206,1070,"[-11.925, -11.925, -11.85, -11.85, -11.875, -1...","[-4.686380092992484, -4.686380092992484, 0.0, ...","{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5...",...,False,2221,GO_R,0832,fi211020a,GO,"[-0.7, -0.7, -0.625, -0.625, -0.675, -0.675, -...","[-3.3080330068182238, -3.3080330068182238, 0.2...",fi211020a.0832,"{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5..."
110355,None,180,L,fi211020a.1304,"[1210, 1287]",8206,1051,"[-11.95, -11.95, -11.95, -11.95, -11.95, -11.9...","[2.389134949368717, 0.5513388344697039, 0.5513...","{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7...",...,False,2202,GO_L,1304,fi211020a,GO,"[0.725, 0.75, 0.75, 0.775, 0.75, 0.75, 0.725, ...","[-1.6540165034091119, -1.1026776689394078, -1....",fi211020a.1304,"{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7..."
110356,None,180,L,fi211020a.0719,"[1123, 1197]",8194,1001,"[0.0, 0.0, -0.1, -0.175, -0.175, -0.175, -0.37...","[-51.090731994192566, -51.090731994192566, -50...","{0: [521.64, 535.58, 677.96, 741.66, 963.88, 9...",...,True,2057,GO_L,0719,fi211020a,GO,"[-4.425, -4.425, -3.35, -3.0, -2.65, -2.65, -1...","[188.098432359914, 188.09843235

In [29]:
# Drop old neural_data and trial_name_y columns, rename new_neural_data to neural_data
merged_df = merged_df.drop(columns=['neural_data', 'trial_name_y'])
merged_df = merged_df.rename(columns={'new_neural_data': 'neural_data'})

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Updated columns: {list(merged_df.columns)}")

merged_df

Updated merged_df shape: (110358, 28)
Updated columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name_x', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel', 'neural_data']


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,reaction_time,...,stop_cue,trial_failed,trial_length,trial_name_x,trial_number,trial_session,type,vPos,vVel,neural_data
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...",328.0,...,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4....","{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [..."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....",186.0,...,NaN,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1...","{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...",161.0,...,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4...","{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...",132.0,...,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ...","{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...",NaN,...,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0...","{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110353,None,0,R,fi211020a.0230,"[1213, 1285]",8206,943,"[-9.925, -9.925, -9.925, -9.925, -9.95, -9.95,...","[-0.09188980574495066, -0.09188980574495066, -...",270.0,...,NaN,False,2094,GO_R,0230,fi211020a,GO,"[-1.35, -1.35, -1.45, -1.45, -1.35, -1.35, -1....","[-1.8377961148990132, -1.8377961148990132, -3....","{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7..."
110354,None,0,R,fi211020a.0832,"[1397, 1470]",8206,1070,"[-11.925, -11.925, -11.85, -11.85, -11.875, -1...","[-4.686380092992484, -4.686380092992484, 0.0, ...",327.0,...,NaN,False,2221,GO_R,0832,fi211020a,GO,"[-0.7, -0.7, -0.625, -0.625, -0.675, -0.675, -...","[-3.3080330068182238, -3.3080330068182238, 0.2...","{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5..."
110355,None,180,L,fi211020a.1304,"[1210, 1287]",8206,1051,"[-11.95, -11.95, -11.95, -11.95, -11.95, -11.9...","[2.389134949368717, 0.5513388344697039, 0.5513...",159.0,...,NaN,False,2202,GO_L,1304,fi211020a,GO,"[0.725, 0.75, 0.75, 0.775, 0.75, 0.75, 0.725, ...","[-1.6540165034091119, -1.1026776689394078, -1....","{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7..."
110356,None,180,L,fi211020a.0719,"[1123, 1197]",8194,1001,"[0.0, 0.0, -0.1, -0.175, -0.175, -0.175, -0.37...","[-51.090731994192566, -51.090731994192566, -50...",122.0,...,NaN,True,2057,GO_L,0719,fi211020a,GO,"[-4.425, -4.425, -3.35, -3.0, -2.65, -2.65, -1...","[188.098432359914, 188.098432359914, 188.09843...","{0: [521.64, 535.58, 677.96, 741.66, 963.88, 9..."


In [30]:
# Rename trial_name_x to trial_name
merged_df = merged_df.rename(columns={'trial_name_x': 'trial_name'})

# Reorder columns to match orig_df
orig_columns = list(orig_df.columns)
merged_df = merged_df[orig_columns]

print(f"Updated merged_df shape: {merged_df.shape}")
print(f"Original df columns: {list(orig_df.columns)}")
print(f"Merged df columns: {list(merged_df.columns)}")
print(f"\nColumns match: {list(orig_df.columns) == list(merged_df.columns)}")

merged_df

Updated merged_df shape: (110358, 28)
Original df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel']
Merged df columns: ['blinks', 'dir', 'direction', 'filename', 'first_relevant_saccade', 'flags', 'go_cue', 'hPos', 'hVel', 'neural_data', 'reaction_time', 'saccades', 'screen_rotation', 'segs_durations', 'segs_times', 'set', 'speed', 'ssd_len', 'ssd_number', 'stop_cue', 'trial_failed', 'trial_length', 'trial_name', 'trial_number', 'trial_session', 'type', 'vPos', 'vVel']

Columns match: True


,blinks,dir,direction,filename,first_relevant_saccade,flags,go_cue,hPos,hVel,neural_data,...,ssd_number,stop_cue,trial_failed,trial_length,trial_name,trial_number,trial_session,type,vPos,vVel
0,None,180,L,fi210824a.0614,"[1382, 1457]",8206,1054,"[11.275, 11.275, 11.275, 11.275, 11.275, 11.27...","[0.0, 0.0, 0.4594490287247533, 1.3783470861742...","{0: [884.4], 1: [154.42, 329.18, 1478.9], 2: [...",...,2.0,1186.0,False,2205,CONT_L_SSD2,0614,fi210824a,CONT,"[-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, 0.0...","[-3.1242533953283225, -3.1242533953283225, -4...."
1,None,180,L,fi210824a.0520,"[1100, 1172]",8206,914,"[-11.15, -11.15, -11.15, -11.15, -11.15, -11.1...","[-2.7566941723485194, -2.7566941723485194, -3....","{0: [], 1: [48.52, 264.55, 585.97, 1032.3], 2:...",...,NaN,NaN,False,2065,GO_L,0520,fi210824a,GO,"[-1.15, -1.15, -1.15, -1.15, -1.15, -1.175, -1...","[-0.8270082517045559, -0.8270082517045559, 0.1..."
2,None,180,L,fi210824a.1193,"[1099, 1179]",8206,938,"[2.25, 2.25, 2.3, 2.3, 2.3, 2.25, 2.225, 2.225...","[-188.19032216565896, -188.19032216565896, -18...","{0: [], 1: [574.7, 853.02, 1403.57], 2: [], 3:...",...,3.0,1118.0,False,2089,CONT_L_SSD3,1193,fi210824a,CONT,"[-27.45, -27.45, -27.45, -27.45, -27.45, -27.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 9.280870380240016, 4..."
3,None,180,L,fi210824a.1013,"[1213, 1289]",13326,1081,"[8.425, 8.425, 8.375, 8.375, 8.375, 8.4, 8.4, ...","[1.286457280429309, 1.286457280429309, -0.9188...","{0: [], 1: [], 2: [1954.73], 3: [], 4: [], 5: ...",...,3.0,1261.0,True,1961,STOP_L_SSD3,1013,fi210824a,STOP,"[0.275, 0.275, 0.275, 0.275, 0.275, 0.275, 0.2...","[0.9188980574495066, 0.9188980574495066, 0.0, ..."
4,None,0,R,fi210824a.1257,NaN,8194,1024,"[-11.475, -11.475, -11.475, -11.475, -11.475, ...","[3.767482035542977, 3.767482035542977, 3.85937...","{0: [923.35], 1: [1125], 2: [], 3: [], 4: [], ...",...,2.0,1156.0,True,1476,CONT_R_SSD2,1257,fi210824a,CONT,"[-1.825, -1.825, -1.8, -1.8, -1.825, -1.825, -...","[0.27566941723485194, 0.27566941723485194, 1.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110353,None,0,R,fi211020a.0230,"[1213, 1285]",8206,943,"[-9.925, -9.925, -9.925, -9.925, -9.95, -9.95,...","[-0.09188980574495066, -0.09188980574495066, -...","{0: [1937.51], 1: [24.51, 42.58, 111.86, 173.7...",...,NaN,NaN,False,2094,GO_R,0230,fi211020a,GO,"[-1.35, -1.35, -1.45, -1.45, -1.35, -1.35, -1....","[-1.8377961148990132, -1.8377961148990132, -3...."
110354,None,0,R,fi211020a.0832,"[1397, 1470]",8206,1070,"[-11.925, -11.925, -11.85, -11.85, -11.875, -1...","[-4.686380092992484, -4.686380092992484, 0.0, ...","{0: [1027.96], 1: [19.86, 91.81, 177.64, 261.5...",...,NaN,NaN,False,2221,GO_R,0832,fi211020a,GO,"[-0.7, -0.7, -0.625, -0.625, -0.675, -0.675, -...","[-3.3080330068182238, -3.3080330068182238, 0.2..."
110355,None,180,L,fi211020a.1304,"[1210, 1287]",8206,1051,"[-11.95, -11.95, -11.95, -11.95, -11.95, -11.9...","[2.389134949368717, 0.5513388344697039, 0.5513...","{0: [16.69, 27.66, 37.08, 89.74, 120.56, 218.7...",...,NaN,NaN,False,2202,GO_L,1304,fi211020a,GO,"[0.725, 0.75, 0.75, 0.775, 0.75, 0.75, 0.725, ...","[-1.6540165034091119, -1.1026776689394078, -1...."
110356,None,180,L,fi211020a.0719,"[1123, 1197]",8194,1001,"[0.0, 0.0, -0.1, -0.175, -0.175, -0.175, -0.37...","[-51.090731994192566, -51.090731994192566, -50...","{0: [521.64, 535.58, 677.96, 741.66, 963.88, 9...",...,NaN,NaN,True,2057,GO_L,0719,fi211020a,GO,"[-4.425, -4.425, -3.35, -3.0, -2.65, -2.65, -1...","[188.098432359914, 188.098432359914, 188.09843..."


In [32]:
neurons_list = []

def extract_neuron_ids(row, neurons_list):
    trial_neuron_ids = [f"{row['trial_session']}_{key}" for key in row['neural_data'].keys()]
    neurons_list += trial_neuron_ids
    return trial_neuron_ids

merged_df.apply(
    lambda row: extract_neuron_ids(row, neurons_list), 
    axis=1
)

neurons_set = set(neurons_list)
print(f'Total unique neurons across all sessions: {len(neurons_set)}')

Total unique neurons across all sessions: 17600


In [ ]:
file_path = base_path / 'csst_trials_pkls' / f'all_{monkey_name}_CSST_trials_df_including_all_200_neuron_fields.pkl'
print(f"Saving updated DataFrame to: {file_path}")

Saving updated DataFrame to: /home/barak/Projects/population-analysis/data/csst_trials_pkls/all_fiona_CSST_trials_df_including_all_200_neuron_fields.pkl


In [ ]:
# merged_df.to_pickle(file_path)

In [26]:
# monkey = 'fiona'
# save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
# pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'

# cells_db = pd.read_pickle(pickle_file)
# cells_db['maestro_ID'].value_counts().sort_index()